In [ ]:
# Run this cell only when you want to make model calls.
# Put this notebook next to the data/prompt files, and sign in to Antigravity CLI.
# IDs in code or report text remain unchanged; separate identifier fields are not sent.
"""B0 prediction and evaluation helpers. Importing this module makes no calls."""
import csv
import hashlib
import json
import math
import os
from pathlib import Path
import shutil
import subprocess
import tempfile
import time
import uuid

PRE_AUTH_FIX_SHA256 = "4ffaacc878b8617d58016be552f2874de6f7544e261cd5846fe98cb7741d06b7"


def interactive_login(cli, reason):
    """User handles OAuth in a real terminal, never in captured subprocess IO."""
    print(f"\n{reason}\nProgress is saved. No requests run while this prompt is waiting.")
    print("Choose login to open a visible Antigravity terminal. To switch accounts, "
          "enter /logout there, then sign in with the other account.")
    print("Paste any browser authorization code/link ONLY into that terminal's "
          "authorization prompt. After sign-in, exit the CLI with /exit.")
    while True:
        try:
            choice = input("Type login, ready (already signed in), or stop: ").strip().lower()
        except (EOFError, KeyboardInterrupt):
            print("Stopped. Sign in with agy in PowerShell, then rerun cell 1.")
            return False
        if choice == "stop":
            return False
        if choice == "ready":
            return True
        if choice == "login":
            if os.name == "nt":
                # Single-quoted PowerShell literal; never interpolate OAuth data.
                quoted_cli = "'" + str(cli).replace("'", "''") + "'"
                try:
                    subprocess.Popen(
                        ["powershell.exe", "-NoProfile", "-NoExit", "-Command", "& " + quoted_cli],
                        cwd=tempfile.gettempdir(), creationflags=subprocess.CREATE_NEW_CONSOLE,
                        stdin=None, stdout=None, stderr=None,
                    )
                except OSError as exc:
                    print(f"Cannot open terminal: {exc}. Open PowerShell and run agy manually.")
            else:
                print(f"Open a separate terminal and run: {cli}")
            print("Complete sign-in there, then return HERE and type ready. "
                  "If /logout closes the CLI, run agy again in that terminal.")
        else:
            print("Enter login, ready, or stop; do not paste authorization credentials here.")

SETTINGS = {
    "toolPermission": "strict", "useG1Credits": False,
    "disableSlashCommands": True, "allowNonWorkspaceAccess": False,
    "permissions": {"allow": [], "ask": [], "deny": [
        "read_url(*)", "execute_url(*)", "command(*)", "mcp(*)",
        "read_file(*)", "write_file(*)"]},
}
SCHEMA = {"type": "object", "properties": {"flaky_probability": {
    "type": "number", "minimum": 0, "maximum": 1}},
    "required": ["flaky_probability"], "additionalProperties": False}


def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False).encode()).hexdigest()


def validate_probability(value):
    if not isinstance(value, dict) or set(value) != {"flaky_probability"}:
        raise ValueError("Expected only flaky_probability")
    p = value["flaky_probability"]
    if isinstance(p, bool) or not isinstance(p, (int, float)) or not math.isfinite(p) or not 0 <= p <= 1:
        raise ValueError("Probability must be a finite number in [0,1]")
    return float(p)


def parse_response(stdout):
    events = [json.loads(line) for line in stdout.splitlines() if line.strip()]
    for event in events:
        step = event.get("step_update", {})
        tool = step.get("tool_info") or {}
        internal_finish = (step.get("step_type") == "tool" and
                           step.get("tool_name") == "finish" and
                           tool.get("name") == "finish" and
                           not step.get("subagent_info"))
        if (step.get("step_type") == "tool" or tool or step.get("subagent_info")) and not internal_finish:
            raise RuntimeError("Tool use detected; prediction rejected")
    results = [e["result"] for e in events if e.get("event") == "result"]
    if len(results) != 1:
        raise ValueError("Expected exactly one final CLI result")
    result = results[0]
    if result.get("status") != "SUCCESS":
        raise RuntimeError(str(result.get("error", result.get("status"))))
    value = result.get("structured_output")
    if value is None:
        value = json.loads(result["response"])
    return validate_probability(value), result


def classify_error(message):
    msg = message.lower()
    if any(s in msg for s in ("quota", "resource_exhausted", "429", "rate limit", "sign in", "sign-in", "logged in", "authentication", "credential", "credit", "authorization code", "unauthenticated")):
        return "blocked"
    if any(s in msg for s in ("timeout", "timed out", "503", "502", "temporarily", "connection reset")):
        return "transient"
    return "error"


def run_predictions(base, selected_ids=("198",), n_runs=1,
                    model="Gemini 3.8 Flash (Medium)", effort="medium",
                    experiment="b0_flash_medium_v1", timeout=300, cli=None,
                    login_before_run=True):
    base = Path(base).resolve()
    if not isinstance(n_runs, int) or isinstance(n_runs, bool) or n_runs < 1:
        raise ValueError("n_runs must be a positive integer")
    if Path(experiment).name != experiment or experiment in ("", ".", ".."):
        raise ValueError("experiment must be a simple folder name")
    data = json.loads((base / "b0_data_v1.json").read_text(encoding="utf-8"))
    by_id = {r["id"]: r for r in data}
    if len(by_id) != len(data):
        raise ValueError("Duplicate input IDs")
    ids = list(by_id) if selected_ids is None else list(dict.fromkeys(map(str, selected_ids)))
    if not ids or set(ids) - set(by_id):
        raise ValueError("Select existing, nonempty IDs")
    prompt = (base / "prompt_b0.txt").read_text(encoding="utf-8")
    if prompt.count("{{TEST_CODE}}") != 1:
        raise ValueError("Prompt must contain exactly one target placeholder")
    cli = cli or shutil.which("agy") or str(Path.home() / "AppData/Local/agy/bin/agy.exe")
    if not Path(cli).is_file():
        raise FileNotFoundError("Antigravity CLI not found; configure CLI_PATH")
    config = {"model": model, "effort": effort, "prompt_sha256": digest(prompt),
              "data_sha256": digest(data), "settings": SETTINGS, "schema": SCHEMA,
              "runner_sha256": hashlib.sha256(Path(__file__).read_bytes()).hexdigest()}
    folder = base / "b0_results" / experiment
    folder.mkdir(parents=True, exist_ok=True)
    manifest = folder / "manifest.json"
    if manifest.exists():
        old = json.loads(manifest.read_text())
        if old != config:
            compatible_auth_fix = (old.get("runner_sha256") == PRE_AUTH_FIX_SHA256 and
                {k: v for k, v in old.items() if k != "runner_sha256"} ==
                {k: v for k, v in config.items() if k != "runner_sha256"})
            if not compatible_auth_fix:
                raise ValueError("Experiment configuration changed: use a new EXPERIMENT name")
            (folder / "manifest_before_auth_fix.json").write_text(json.dumps(old, indent=2), encoding="utf-8")
    manifest.write_text(json.dumps(config, indent=2), encoding="utf-8")
    (folder / "prompt_snapshot.txt").write_text(prompt, encoding="utf-8")
    log = folder / "predictions.jsonl"
    previous = [json.loads(line) for line in log.read_text(encoding="utf-8").splitlines()] if log.exists() else []
    done = {(r["run"], r["id"]) for r in previous if r["status"] == "success"}
    if all((run, ident) in done for run in range(1, n_runs + 1) for ident in ids):
        return folder
    if login_before_run and not interactive_login(cli, "Confirm interactive sign-in before starting predictions."):
        return folder
    auth_session = uuid.uuid4().hex  # Local audit marker, not an account identifier.
    # One NDJSON input message per fresh process avoids Windows command-length limits.
    for run in range(1, n_runs + 1):
        for ident in ids:
            if (run, ident) in done:
                continue
            target_prompt = prompt.replace("{{TEST_CODE}}", by_id[ident]["test_code"])
            attempt = 0
            transient_failures = 0
            while True:
                attempt += 1
                record = {"run": run, "id": ident, "model": model, "effort": effort,
                          "auth_session": auth_session,
                          "prompt_sha256": digest(target_prompt), "attempt": attempt,
                          "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
                start = time.monotonic()
                try:
                    with tempfile.TemporaryDirectory(prefix="b0_inference_") as work:
                        command = [cli, "--input-format", "stream-json", "--output-format", "stream-json",
                                   "--model", model, "--disable-slash-commands",
                                   "--settings", json.dumps(SETTINGS), "--json-schema", json.dumps(SCHEMA),
                                   "--print-timeout", f"{timeout}s"]
                        if not model.endswith("(Medium)"):
                            command.extend(["--effort", effort])
                        elif effort != "medium":
                            raise ValueError("Medium model requires medium effort")
                        completed = subprocess.run(command, input=json.dumps({"event": "user", "message": {
                            "content": target_prompt}}) + "\n", capture_output=True, text=True,
                            encoding="utf-8", errors="replace", cwd=work, timeout=timeout + 30,
                            creationflags=getattr(subprocess, "CREATE_NO_WINDOW", 0))
                    if completed.returncode:
                        if classify_error(completed.stderr + "\n" + completed.stdout) == "blocked":
                            raise RuntimeError("Authentication or account quota/credits require attention")
                        raise RuntimeError((completed.stderr + "\n" + completed.stdout)[-6000:])
                    probability, result = parse_response(completed.stdout)
                    record.update(status="success", flaky_probability=probability,
                                  conversation_id=result.get("conversation_id"), usage=result.get("usage"))
                except (OSError, subprocess.SubprocessError, ValueError, RuntimeError) as exc:
                    record.update(status=classify_error(str(exc)), error=str(exc))
                record["duration_seconds"] = round(time.monotonic() - start, 3)
                with log.open("a", encoding="utf-8") as f:
                    f.write(json.dumps(record, ensure_ascii=False) + "\n")
                    f.flush()
                    os.fsync(f.fileno())
                print(f"Run {run}, ID {ident}: {record['status']} "
                      f"{record.get('flaky_probability', '')}")
                if record["status"] == "blocked":
                    if not interactive_login(cli, "Authentication failed or account quota/credits are exhausted."):
                        return folder
                    auth_session = uuid.uuid4().hex
                    transient_failures = 0
                    continue  # Retry this exact row; never discard it or restart completed work.
                if record["status"] != "transient":
                    break
                transient_failures += 1
                if transient_failures >= 3:
                    break
                time.sleep(2 ** transient_failures)
    return folder


def compare_results(base, experiment, threshold=0.5):
    import pandas as pd
    from sklearn.metrics import accuracy_score, brier_score_loss, confusion_matrix, precision_recall_fscore_support
    base = Path(base)
    folder = base / "b0_results" / experiment
    path = folder / "predictions.jsonl"
    if not path.exists():
        raise FileNotFoundError("No saved predictions. Run cell 1 when ready.")
    records = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines()]
    # Prefer a completed success; otherwise report the latest failed attempt.
    latest = {}
    for r in records:
        key = (r["run"], r["id"])
        if key not in latest or latest[key]["status"] != "success":
            latest[key] = r
    predictions = pd.DataFrame(latest.values())
    labels = pd.read_csv(base / "sample_300_labels.csv", dtype={"id": str})[["id", "fleaky"]]
    labels = labels.rename(columns={"fleaky": "actual"})
    if not labels.actual.isin([0, 1]).all():
        raise ValueError("Invalid ground-truth labels")
    table = predictions.merge(labels, on="id", how="left", validate="many_to_one")
    if table.actual.isna().any():
        raise ValueError("Prediction IDs missing from labels")
    if "flaky_probability" not in table:
        table["flaky_probability"] = float("nan")
    example_ids = {r["id"] for r in json.loads((base / "b0_few_shot.json").read_text(encoding="utf-8"))}
    table["few_shot"] = table.id.isin(example_ids)
    table["predicted"] = table.flaky_probability.ge(threshold).astype("Int64")
    table.loc[table.status != "success", "predicted"] = pd.NA
    table["correct"] = table.predicted.eq(table.actual)
    summaries = []
    for run, group in table.groupby("run"):
        for scope in ("all", "excluding_few_shot"):
            sub = group if scope == "all" else group[~group.few_shot]
            good = sub[sub.status == "success"]
            row = {"run": run, "scope": scope, "successful": len(good),
                   "failed": len(sub) - len(good), "not_attempted": len(labels) - len(group)}
            if len(good):
                y, pred = good.actual.astype(int), good.predicted.astype(int)
                precision, recall, f1, support = precision_recall_fscore_support(y, pred, labels=[0, 1], zero_division=0)
                row.update(accuracy=accuracy_score(y, pred), brier=brier_score_loss(y, good.flaky_probability))
                for k, name in enumerate(("non_flaky", "flaky")):
                    row.update({f"{name}_precision": precision[k], f"{name}_recall": recall[k],
                                f"{name}_f1": f1[k], f"{name}_support": int(support[k])})
                row.update(zip(("tn", "fp", "fn", "tp"), confusion_matrix(y, pred, labels=[0, 1]).ravel().tolist()))
            summaries.append(row)
    metrics = pd.DataFrame(summaries)
    variation = table[table.status == "success"].groupby("id").flaky_probability.agg(["count", "mean", "std", "min", "max"])
    table.to_csv(folder / "comparison.csv", index=False)
    metrics.to_csv(folder / "metrics.csv", index=False)
    variation.to_csv(folder / "probability_variability.csv")
    aggregate = metrics.groupby("scope").agg({c: ["mean", "std"] for c in metrics if c not in {"run", "scope"}})
    aggregate.to_csv(folder / "metrics_across_runs.csv")
    return table, metrics, aggregate, variation


"""Six independent CLI lanes with a single checkpoint writer and sign-in gate."""
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED
from collections import deque
from pathlib import Path
import json
import os
import shutil
import subprocess
import tempfile
import time
import uuid



def request_one(cli, prompt, model, effort, timeout):
    start = time.monotonic()
    try:
        with tempfile.TemporaryDirectory(prefix="b0_inference_") as work:
            command = [cli, "--input-format", "stream-json", "--output-format", "stream-json",
                       "--model", model, "--disable-slash-commands", "--settings", json.dumps(SETTINGS),
                       "--json-schema", json.dumps(SCHEMA), "--print-timeout", f"{timeout}s"]
            if not model.endswith("(Medium)"):
                command.extend(["--effort", effort])
            elif effort != "medium":
                raise ValueError("Medium model requires medium effort")
            completed = subprocess.run(command, input=json.dumps({"event": "user", "message": {
                "content": prompt}}) + "\n", capture_output=True, text=True, encoding="utf-8",
                errors="replace", cwd=work, timeout=timeout + 30,
                creationflags=getattr(subprocess, "CREATE_NO_WINDOW", 0))
        if completed.returncode:
            message = completed.stderr + "\n" + completed.stdout
            if classify_error(message) == "blocked":
                raise RuntimeError("Authentication or quota/credits require attention")
            raise RuntimeError(message[-4000:])
        probability, result = parse_response(completed.stdout)
        record = dict(status="success", flaky_probability=probability,
                      conversation_id=result.get("conversation_id"), usage=result.get("usage"))
    except (OSError, subprocess.SubprocessError, ValueError, RuntimeError) as exc:
        record = dict(status=classify_error(str(exc)), error=str(exc))
    record["duration_seconds"] = round(time.monotonic() - start, 3)
    return record


def run_parallel(base, selected_ids=None, n_runs=3, workers=6,
                 model="Gemini 3.8 Flash (Medium)", effort="medium",
                 experiment="b0_flash_medium_v3", timeout=300, cli=None,
                 login_before_run=False, login_handler=interactive_login, request_fn=request_one):
    base = Path(base).resolve()
    if workers < 1 or n_runs < 1:
        raise ValueError("workers and n_runs must be positive")
    if Path(experiment).name != experiment or experiment in ("", ".", ".."):
        raise ValueError("Invalid experiment folder")
    data = json.loads((base / "b0_data_v1.json").read_text(encoding="utf-8"))
    by_id = {r["id"]: r for r in data}
    ids = list(by_id) if selected_ids is None else list(dict.fromkeys(map(str, selected_ids)))
    if not ids or set(ids) - set(by_id) or len(by_id) != len(data):
        raise ValueError("Invalid/duplicate input IDs")
    prompt = (base / "prompt_b0.txt").read_text(encoding="utf-8")
    if prompt.count("{{TEST_CODE}}") != 1:
        raise ValueError("Expected one target placeholder")
    cli = cli or shutil.which("agy") or str(Path.home() / "AppData/Local/agy/bin/agy.exe")
    folder = base / "b0_results" / experiment
    folder.mkdir(parents=True, exist_ok=True)
    # OS-held lock is released even after crashes. Never run two schedulers together.
    lock_file = (folder / "parallel.lock").open("a+b")
    lock_file.seek(0)
    if os.name == "nt":
        import msvcrt
        try:
            msvcrt.locking(lock_file.fileno(), msvcrt.LK_NBLCK, 1)
        except OSError:
            lock_file.close()
            raise RuntimeError("Another parallel runner already holds the experiment lock")
    try:
        config = dict(model=model, effort=effort, prompt_sha256=digest(prompt), data_sha256=digest(data),
                      settings=SETTINGS, schema=SCHEMA)
        manifest = folder / "manifest.json"
        if manifest.exists():
            old = json.loads(manifest.read_text())
            if any(old.get(k) != v for k, v in config.items()):
                raise ValueError("Prompt/model/data changed; use a new experiment")
        else:
            manifest.write_text(json.dumps(config, indent=2))
        (folder / "parallel_config.json").write_text(json.dumps(dict(workers=workers, n_runs=n_runs,
            selected_ids=ids, scheduler_sha256=digest("embedded-notebook-scheduler-v1")), indent=2))
        log = folder / "predictions.jsonl"
        previous = [json.loads(s) for s in log.read_text(encoding="utf-8").splitlines()] if log.exists() else []
        done = {(r["run"], r["id"]) for r in previous if r["status"] == "success"}
        assignments = [[(run, ident) for run in range(1, n_runs + 1) for ident in ids[w::workers]] for w in range(workers)]
        queues = [deque(job for job in jobs if job not in done) for jobs in assignments]
        (folder / "worker_assignments.json").write_text(json.dumps([
            {"worker": w + 1, "assigned": len(jobs), "pending": len(queues[w]), "jobs": jobs}
            for w, jobs in enumerate(assignments)], indent=2))
        auth_session = uuid.uuid4().hex
        paused = False
        stopped = False
        attempts = {}
        retry_at = {}
        pending = {}

        def status(state):
            payload = dict(state=state, workers=workers, active=len(pending),
                remaining=sum(map(len, queues)), completed_successes=len(done),
                updated_utc=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()))
            temp = folder / "parallel_status.tmp"
            temp.write_text(json.dumps(payload, indent=2))
            temp.replace(folder / "parallel_status.json")

        if login_before_run and not login_handler(cli, "Sign in before starting six workers."):
            status("stopped")
            return folder
        with ThreadPoolExecutor(max_workers=workers) as pool:
            while any(queues) or pending:
                if paused and not pending:
                    status("awaiting_login")
                    print("ALL WORKERS PAUSED: quota/authentication. No requests are in flight.", flush=True)
                    if not login_handler(cli, "Sign in with the desired account, then choose ready to resume all six workers."):
                        stopped = True
                        break
                    auth_session = uuid.uuid4().hex
                    paused = False
                if not paused:
                    busy = {w for w, _ in pending.values()}
                    for w, queue in enumerate(queues):
                        if queue and w not in busy and time.monotonic() >= retry_at.get(w, 0):
                            run, ident = queue[0]
                            target = prompt.replace("{{TEST_CODE}}", by_id[ident]["test_code"])
                            record = dict(run=run, id=ident, worker=w + 1, model=model, effort=effort,
                                auth_session=auth_session, prompt_sha256=digest(target),
                                timestamp_utc=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()))
                            future = pool.submit(request_fn, cli, target, model, effort, timeout)
                            pending[future] = (w, record)
                status("draining_for_login" if paused else "running")
                if not pending:
                    time.sleep(0.2)
                    continue
                finished, _ = wait(pending, timeout=1, return_when=FIRST_COMPLETED)
                for future in finished:
                    w, record = pending.pop(future)
                    record.update(future.result())
                    key = (record["run"], record["id"])
                    attempts[key] = attempts.get(key, 0) + 1
                    record["attempt"] = attempts[key]
                    with log.open("a", encoding="utf-8") as f:
                        f.write(json.dumps(record, ensure_ascii=False) + "\n")
                        f.flush()
                        os.fsync(f.fileno())
                    print(f"Worker {w+1}: run {key[0]}, ID {key[1]}: {record['status']} "
                          f"{record.get('flaky_probability', '')}", flush=True)
                    if record["status"] == "blocked":
                        paused = True  # Drain existing calls; never issue new ones until re-login.
                        attempts[key] = 0
                    elif record["status"] == "transient" and attempts[key] < 3:
                        retry_at[w] = time.monotonic() + 2 ** attempts[key]
                    else:
                        queues[w].popleft()
                        if record["status"] == "success":
                            done.add(key)
            status("stopped" if stopped else "finished")
        return folder
    finally:
        lock_file.close()



# Recursive retry controller: keeps successful predictions and retries unresolved IDs.
def run_until_complete(base, ids, n_runs, workers, experiment, login_before_run):
    dataset = json.loads((base / DATA_FILE).read_text(encoding="utf-8"))
    known = {str(r["id"]) for r in dataset}
    selected = list(dict.fromkeys(map(str, ids))) if ids else [str(r["id"]) for r in dataset]
    unknown = set(selected) - known
    if unknown:
        raise ValueError(f"Unknown IDs: {sorted(unknown)}")
    expected = {(n, ident) for n in range(1, n_runs + 1) for ident in selected}
    folder = base / RESULTS_DIRNAME / experiment
    log = folder / "predictions.jsonl"
    pass_number = 0
    while True:
        records = [json.loads(line) for line in log.read_text(encoding="utf-8").splitlines() if line.strip()] if log.exists() else []
        completed = {(int(r["run"]), str(r["id"])) for r in records if r["status"] == "success"}
        remaining = expected - completed
        if not remaining:
            print(f"Complete: {len(expected)} selected predictions have valid results.")
            return folder
        pass_number += 1
        print(f"Pass {pass_number}: {len(remaining)} predictions remaining.", flush=True)
        run_parallel(base, selected_ids=selected, n_runs=n_runs, workers=workers,
                     experiment=experiment, model=MODEL, effort="medium",
                     login_before_run=login_before_run and pass_number == 1)
        state = json.loads((folder / "parallel_status.json").read_text(encoding="utf-8"))
        if state["state"] == "stopped":
            print("Stopped; rerun the cell to resume from saved results.")
            return folder
        records = [json.loads(line) for line in log.read_text(encoding="utf-8").splitlines() if line.strip()] if log.exists() else []
        completed = {(int(r["run"]), str(r["id"])) for r in records if r["status"] == "success"}
        if expected - completed:
            delay = min(60, pass_number * 10)
            print(f"Retrying remaining predictions in {delay} seconds.", flush=True)
            time.sleep(delay)

# --- B0 configuration ---
BASE = Path.cwd()
B0_IDS = []          # [] = all 300; e.g. ["198", "200"] = only these IDs
B0_N_RUNS = 3
B0_WORKERS = 8
B0_EXPERIMENT = "b0_flash_medium_v3"
MODEL = "Gemini 3.8 Flash (Medium)"
LOGIN_BEFORE_RUN = False  # True opens a login menu before the first call.
DATA_FILE = "b0_data_v1.json"
RESULTS_DIRNAME = "b0_results"
B0_RESULTS_DIR = run_until_complete(BASE, B0_IDS, B0_N_RUNS, B0_WORKERS, B0_EXPERIMENT, LOGIN_BEFORE_RUN)


In [1]:
# Compare saved predictions to actual labels. This cell makes no model calls.
from pathlib import Path
import json
import pandas as pd
from IPython.display import display
from sklearn.metrics import accuracy_score, brier_score_loss, confusion_matrix, precision_recall_fscore_support

def show_results(level, experiment, selected_ids, threshold=0.5):
    base = Path.cwd()
    folder = base / (level.lower() + "_results") / experiment
    log = folder / "predictions.jsonl"
    if not log.exists():
        raise FileNotFoundError(f"No saved predictions: {log}")
    latest = {}
    for line in log.read_text(encoding="utf-8").splitlines():
        r = json.loads(line)
        key = (int(r["run"]), str(r["id"]))
        if key not in latest or latest[key]["status"] != "success":
            latest[key] = r
    frame = pd.DataFrame(latest.values())
    frame["id"] = frame.id.astype(str)
    if selected_ids:
        frame = frame[frame.id.isin(list(map(str, selected_ids)))].copy()
    if frame.empty:
        print("No predictions found for the selected IDs.")
        return
    labels = pd.read_csv(base / "sample_300_labels.csv", dtype={"id": str})[["id", "fleaky"]].rename(columns={"fleaky": "actual"})
    frame = frame.merge(labels, on="id", how="left", validate="many_to_one")
    if frame.actual.isna().any():
        raise ValueError("A prediction has no matching label")
    if "flaky_probability" not in frame:
        frame["flaky_probability"] = float("nan")
    frame["predicted"] = frame.flaky_probability.ge(threshold).astype("Int64")
    frame.loc[frame.status != "success", "predicted"] = pd.NA
    frame["correct"] = frame.predicted.eq(frame.actual)
    rows = []
    for run, group in frame.groupby("run"):
        good = group[group.status == "success"]
        row = {"run": run, "successes": len(good), "failures": len(group)-len(good)}
        if len(good):
            y, pred = good.actual.astype(int), good.predicted.astype(int)
            precision, recall, f1, _ = precision_recall_fscore_support(y, pred, labels=[0,1], zero_division=0)
            tn, fp, fn, tp = confusion_matrix(y,pred,labels=[0,1]).ravel()
            row.update(accuracy=accuracy_score(y,pred), brier=brier_score_loss(y,good.flaky_probability),
                       flaky_precision=precision[1], flaky_recall=recall[1], flaky_f1=f1[1],
                       non_flaky_precision=precision[0], non_flaky_recall=recall[0], non_flaky_f1=f1[0],
                       tn=tn, fp=fp, fn=fn, tp=tp)
        rows.append(row)
    print(f"{level} metrics, threshold >= {threshold}; all selected tests including few-shot examples")
    display(pd.DataFrame(rows))
    print("Per-test results:")
    display(frame[["run", "id", "actual", "flaky_probability", "predicted", "correct", "status"]])
    valid = frame[frame.status == "success"]
    if len(valid):
        wide = valid.pivot(index="id", columns="run", values="flaky_probability")
        all_runs = wide.dropna()
        if len(wide.columns) >= 2 and len(all_runs):
            labels_by_run = all_runs.ge(threshold)
            agreement = labels_by_run.eq(labels_by_run.iloc[:,0],axis=0).all(axis=1)
            print(f"Same class across all available complete runs: {agreement.sum()}/{len(agreement)} ({agreement.mean():.1%})")
            print(f"Mean probability range: {(all_runs.max(axis=1)-all_runs.min(axis=1)).mean():.3f}")
        print("Probability variability by test:")
        display(valid.groupby("id").flaky_probability.agg(["count","mean","std","min","max"]))

B0_RESULTS_TO_COMPARE = "b0_flash_medium_v2"
B0_IDS = globals().get("B0_IDS", [])
B0_THRESHOLD = 0.5
show_results("B0", B0_RESULTS_TO_COMPARE, B0_IDS, B0_THRESHOLD)


B0 metrics, threshold >= 0.5; all selected tests including few-shot examples


,run,successes,failures,accuracy,brier,flaky_precision,flaky_recall,flaky_f1,non_flaky_precision,non_flaky_recall,non_flaky_f1,tn,fp,fn,tp
0,1,300,0,0.503333,0.324197,0.536585,0.546584,0.541538,0.463235,0.453237,0.458182,63,76,73,88
1,2,300,0,0.513333,0.313735,0.546012,0.552795,0.549383,0.474453,0.467626,0.471014,65,74,72,89
2,3,300,0,0.503333,0.315074,0.536145,0.552795,0.544343,0.462687,0.446043,0.454212,62,77,72,89


Per-test results:


,run,id,actual,flaky_probability,predicted,correct,status
0,1,198,0,0.90,1,False,success
1,2,198,0,0.90,1,False,success
2,3,198,0,0.95,1,False,success
3,1,200,0,0.95,1,False,success
4,1,262,0,0.85,1,False,success
...,...,...,...,...,...,...,...
895,3,627,1,0.35,0,False,success
896,3,47,1,0.85,1,True,success
897,3,844,1,0.20,0,False,success
898,3,207,1,0.70,1,True,success


Same class across all available complete runs: 254/300 (84.7%)
Mean probability range: 0.113
Probability variability by test:


,count,mean,std,min,max
id,,,,,
110,3,0.883333,0.028868,0.85,0.90
113,3,0.216667,0.028868,0.20,0.25
125,3,0.850000,0.000000,0.85,0.85
13,3,0.466667,0.256580,0.25,0.75
130,3,0.850000,0.000000,0.85,0.85
...,...,...,...,...,...
952,3,0.900000,0.000000,0.90,0.90
953,3,0.950000,0.000000,0.95,0.95
954,3,0.883333,0.028868,0.85,0.90


In [ ]:
# Run this cell only when you want to make model calls.
# Put this notebook next to the data/prompt files, and sign in to Antigravity CLI.
# IDs in code or report text remain unchanged; separate identifier fields are not sent.
"""B0 prediction and evaluation helpers. Importing this module makes no calls."""
import csv
import hashlib
import json
import math
import os
from pathlib import Path
import shutil
import subprocess
import tempfile
import time
import uuid

PRE_AUTH_FIX_SHA256 = "4ffaacc878b8617d58016be552f2874de6f7544e261cd5846fe98cb7741d06b7"


def interactive_login(cli, reason):
    """User handles OAuth in a real terminal, never in captured subprocess IO."""
    print(f"\n{reason}\nProgress is saved. No requests run while this prompt is waiting.")
    print("Choose login to open a visible Antigravity terminal. To switch accounts, "
          "enter /logout there, then sign in with the other account.")
    print("Paste any browser authorization code/link ONLY into that terminal's "
          "authorization prompt. After sign-in, exit the CLI with /exit.")
    while True:
        try:
            choice = input("Type login, ready (already signed in), or stop: ").strip().lower()
        except (EOFError, KeyboardInterrupt):
            print("Stopped. Sign in with agy in PowerShell, then rerun cell 1.")
            return False
        if choice == "stop":
            return False
        if choice == "ready":
            return True
        if choice == "login":
            if os.name == "nt":
                # Single-quoted PowerShell literal; never interpolate OAuth data.
                quoted_cli = "'" + str(cli).replace("'", "''") + "'"
                try:
                    subprocess.Popen(
                        ["powershell.exe", "-NoProfile", "-NoExit", "-Command", "& " + quoted_cli],
                        cwd=tempfile.gettempdir(), creationflags=subprocess.CREATE_NEW_CONSOLE,
                        stdin=None, stdout=None, stderr=None,
                    )
                except OSError as exc:
                    print(f"Cannot open terminal: {exc}. Open PowerShell and run agy manually.")
            else:
                print(f"Open a separate terminal and run: {cli}")
            print("Complete sign-in there, then return HERE and type ready. "
                  "If /logout closes the CLI, run agy again in that terminal.")
        else:
            print("Enter login, ready, or stop; do not paste authorization credentials here.")

SETTINGS = {
    "toolPermission": "strict", "useG1Credits": False,
    "disableSlashCommands": True, "allowNonWorkspaceAccess": False,
    "permissions": {"allow": [], "ask": [], "deny": [
        "read_url(*)", "execute_url(*)", "command(*)", "mcp(*)",
        "read_file(*)", "write_file(*)"]},
}
SCHEMA = {"type": "object", "properties": {"flaky_probability": {
    "type": "number", "minimum": 0, "maximum": 1}},
    "required": ["flaky_probability"], "additionalProperties": False}


def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False).encode()).hexdigest()


def validate_probability(value):
    if not isinstance(value, dict) or set(value) != {"flaky_probability"}:
        raise ValueError("Expected only flaky_probability")
    p = value["flaky_probability"]
    if isinstance(p, bool) or not isinstance(p, (int, float)) or not math.isfinite(p) or not 0 <= p <= 1:
        raise ValueError("Probability must be a finite number in [0,1]")
    return float(p)


def parse_response(stdout):
    events = [json.loads(line) for line in stdout.splitlines() if line.strip()]
    for event in events:
        step = event.get("step_update", {})
        tool = step.get("tool_info") or {}
        internal_finish = (step.get("step_type") == "tool" and
                           step.get("tool_name") == "finish" and
                           tool.get("name") == "finish" and
                           not step.get("subagent_info"))
        if (step.get("step_type") == "tool" or tool or step.get("subagent_info")) and not internal_finish:
            raise RuntimeError("Tool use detected; prediction rejected")
    results = [e["result"] for e in events if e.get("event") == "result"]
    if len(results) != 1:
        raise ValueError("Expected exactly one final CLI result")
    result = results[0]
    if result.get("status") != "SUCCESS":
        raise RuntimeError(str(result.get("error", result.get("status"))))
    value = result.get("structured_output")
    if value is None:
        value = json.loads(result["response"])
    return validate_probability(value), result


def classify_error(message):
    msg = message.lower()
    if any(s in msg for s in ("quota", "resource_exhausted", "429", "rate limit", "sign in", "sign-in", "logged in", "authentication", "credential", "credit", "authorization code", "unauthenticated")):
        return "blocked"
    if any(s in msg for s in ("timeout", "timed out", "503", "502", "temporarily", "connection reset")):
        return "transient"
    return "error"


def run_predictions(base, selected_ids=("198",), n_runs=1,
                    model="Gemini 3.8 Flash (Medium)", effort="medium",
                    experiment="b0_flash_medium_v1", timeout=300, cli=None,
                    login_before_run=True):
    base = Path(base).resolve()
    if not isinstance(n_runs, int) or isinstance(n_runs, bool) or n_runs < 1:
        raise ValueError("n_runs must be a positive integer")
    if Path(experiment).name != experiment or experiment in ("", ".", ".."):
        raise ValueError("experiment must be a simple folder name")
    data = json.loads((base / "b0_data_v1.json").read_text(encoding="utf-8"))
    by_id = {r["id"]: r for r in data}
    if len(by_id) != len(data):
        raise ValueError("Duplicate input IDs")
    ids = list(by_id) if selected_ids is None else list(dict.fromkeys(map(str, selected_ids)))
    if not ids or set(ids) - set(by_id):
        raise ValueError("Select existing, nonempty IDs")
    prompt = (base / "prompt_b0.txt").read_text(encoding="utf-8")
    if prompt.count("{{TEST_CODE}}") != 1:
        raise ValueError("Prompt must contain exactly one target placeholder")
    cli = cli or shutil.which("agy") or str(Path.home() / "AppData/Local/agy/bin/agy.exe")
    if not Path(cli).is_file():
        raise FileNotFoundError("Antigravity CLI not found; configure CLI_PATH")
    config = {"model": model, "effort": effort, "prompt_sha256": digest(prompt),
              "data_sha256": digest(data), "settings": SETTINGS, "schema": SCHEMA,
              "runner_sha256": hashlib.sha256(Path(__file__).read_bytes()).hexdigest()}
    folder = base / "b0_results" / experiment
    folder.mkdir(parents=True, exist_ok=True)
    manifest = folder / "manifest.json"
    if manifest.exists():
        old = json.loads(manifest.read_text())
        if old != config:
            compatible_auth_fix = (old.get("runner_sha256") == PRE_AUTH_FIX_SHA256 and
                {k: v for k, v in old.items() if k != "runner_sha256"} ==
                {k: v for k, v in config.items() if k != "runner_sha256"})
            if not compatible_auth_fix:
                raise ValueError("Experiment configuration changed: use a new EXPERIMENT name")
            (folder / "manifest_before_auth_fix.json").write_text(json.dumps(old, indent=2), encoding="utf-8")
    manifest.write_text(json.dumps(config, indent=2), encoding="utf-8")
    (folder / "prompt_snapshot.txt").write_text(prompt, encoding="utf-8")
    log = folder / "predictions.jsonl"
    previous = [json.loads(line) for line in log.read_text(encoding="utf-8").splitlines()] if log.exists() else []
    done = {(r["run"], r["id"]) for r in previous if r["status"] == "success"}
    if all((run, ident) in done for run in range(1, n_runs + 1) for ident in ids):
        return folder
    if login_before_run and not interactive_login(cli, "Confirm interactive sign-in before starting predictions."):
        return folder
    auth_session = uuid.uuid4().hex  # Local audit marker, not an account identifier.
    # One NDJSON input message per fresh process avoids Windows command-length limits.
    for run in range(1, n_runs + 1):
        for ident in ids:
            if (run, ident) in done:
                continue
            target_prompt = prompt.replace("{{TEST_CODE}}", by_id[ident]["test_code"])
            attempt = 0
            transient_failures = 0
            while True:
                attempt += 1
                record = {"run": run, "id": ident, "model": model, "effort": effort,
                          "auth_session": auth_session,
                          "prompt_sha256": digest(target_prompt), "attempt": attempt,
                          "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
                start = time.monotonic()
                try:
                    with tempfile.TemporaryDirectory(prefix="b0_inference_") as work:
                        command = [cli, "--input-format", "stream-json", "--output-format", "stream-json",
                                   "--model", model, "--disable-slash-commands",
                                   "--settings", json.dumps(SETTINGS), "--json-schema", json.dumps(SCHEMA),
                                   "--print-timeout", f"{timeout}s"]
                        if not model.endswith("(Medium)"):
                            command.extend(["--effort", effort])
                        elif effort != "medium":
                            raise ValueError("Medium model requires medium effort")
                        completed = subprocess.run(command, input=json.dumps({"event": "user", "message": {
                            "content": target_prompt}}) + "\n", capture_output=True, text=True,
                            encoding="utf-8", errors="replace", cwd=work, timeout=timeout + 30,
                            creationflags=getattr(subprocess, "CREATE_NO_WINDOW", 0))
                    if completed.returncode:
                        if classify_error(completed.stderr + "\n" + completed.stdout) == "blocked":
                            raise RuntimeError("Authentication or account quota/credits require attention")
                        raise RuntimeError((completed.stderr + "\n" + completed.stdout)[-6000:])
                    probability, result = parse_response(completed.stdout)
                    record.update(status="success", flaky_probability=probability,
                                  conversation_id=result.get("conversation_id"), usage=result.get("usage"))
                except (OSError, subprocess.SubprocessError, ValueError, RuntimeError) as exc:
                    record.update(status=classify_error(str(exc)), error=str(exc))
                record["duration_seconds"] = round(time.monotonic() - start, 3)
                with log.open("a", encoding="utf-8") as f:
                    f.write(json.dumps(record, ensure_ascii=False) + "\n")
                    f.flush()
                    os.fsync(f.fileno())
                print(f"Run {run}, ID {ident}: {record['status']} "
                      f"{record.get('flaky_probability', '')}")
                if record["status"] == "blocked":
                    if not interactive_login(cli, "Authentication failed or account quota/credits are exhausted."):
                        return folder
                    auth_session = uuid.uuid4().hex
                    transient_failures = 0
                    continue  # Retry this exact row; never discard it or restart completed work.
                if record["status"] != "transient":
                    break
                transient_failures += 1
                if transient_failures >= 3:
                    break
                time.sleep(2 ** transient_failures)
    return folder


def compare_results(base, experiment, threshold=0.5):
    import pandas as pd
    from sklearn.metrics import accuracy_score, brier_score_loss, confusion_matrix, precision_recall_fscore_support
    base = Path(base)
    folder = base / "b0_results" / experiment
    path = folder / "predictions.jsonl"
    if not path.exists():
        raise FileNotFoundError("No saved predictions. Run cell 1 when ready.")
    records = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines()]
    # Prefer a completed success; otherwise report the latest failed attempt.
    latest = {}
    for r in records:
        key = (r["run"], r["id"])
        if key not in latest or latest[key]["status"] != "success":
            latest[key] = r
    predictions = pd.DataFrame(latest.values())
    labels = pd.read_csv(base / "sample_300_labels.csv", dtype={"id": str})[["id", "fleaky"]]
    labels = labels.rename(columns={"fleaky": "actual"})
    if not labels.actual.isin([0, 1]).all():
        raise ValueError("Invalid ground-truth labels")
    table = predictions.merge(labels, on="id", how="left", validate="many_to_one")
    if table.actual.isna().any():
        raise ValueError("Prediction IDs missing from labels")
    if "flaky_probability" not in table:
        table["flaky_probability"] = float("nan")
    example_ids = {r["id"] for r in json.loads((base / "b0_few_shot.json").read_text(encoding="utf-8"))}
    table["few_shot"] = table.id.isin(example_ids)
    table["predicted"] = table.flaky_probability.ge(threshold).astype("Int64")
    table.loc[table.status != "success", "predicted"] = pd.NA
    table["correct"] = table.predicted.eq(table.actual)
    summaries = []
    for run, group in table.groupby("run"):
        for scope in ("all", "excluding_few_shot"):
            sub = group if scope == "all" else group[~group.few_shot]
            good = sub[sub.status == "success"]
            row = {"run": run, "scope": scope, "successful": len(good),
                   "failed": len(sub) - len(good), "not_attempted": len(labels) - len(group)}
            if len(good):
                y, pred = good.actual.astype(int), good.predicted.astype(int)
                precision, recall, f1, support = precision_recall_fscore_support(y, pred, labels=[0, 1], zero_division=0)
                row.update(accuracy=accuracy_score(y, pred), brier=brier_score_loss(y, good.flaky_probability))
                for k, name in enumerate(("non_flaky", "flaky")):
                    row.update({f"{name}_precision": precision[k], f"{name}_recall": recall[k],
                                f"{name}_f1": f1[k], f"{name}_support": int(support[k])})
                row.update(zip(("tn", "fp", "fn", "tp"), confusion_matrix(y, pred, labels=[0, 1]).ravel().tolist()))
            summaries.append(row)
    metrics = pd.DataFrame(summaries)
    variation = table[table.status == "success"].groupby("id").flaky_probability.agg(["count", "mean", "std", "min", "max"])
    table.to_csv(folder / "comparison.csv", index=False)
    metrics.to_csv(folder / "metrics.csv", index=False)
    variation.to_csv(folder / "probability_variability.csv")
    aggregate = metrics.groupby("scope").agg({c: ["mean", "std"] for c in metrics if c not in {"run", "scope"}})
    aggregate.to_csv(folder / "metrics_across_runs.csv")
    return table, metrics, aggregate, variation


EXAMPLES = ("516", "404")
def build_prompt(data, template, target_id):
    records = {r['id']: r for r in data}
    def model_data(ident):
        r = records[ident]
        return json.dumps({'test_code': r['test_code'], 'failure_reports': r['failure_reports']}, ensure_ascii=False, indent=2)
    for token, ident in zip(('{{EXAMPLE_NON_FLAKY_JSON}}', '{{EXAMPLE_FLAKY_JSON}}', '{{TARGET_B1_JSON}}'), (*EXAMPLES, str(target_id))):
        if template.count(token) != 1:
            raise ValueError('Prompt placeholder missing or duplicated: ' + token)
        template = template.replace(token, model_data(ident), 1)
    return template

"""Six independent CLI lanes with a single checkpoint writer and sign-in gate."""
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED
from collections import deque
from pathlib import Path
import json
import os
import shutil
import subprocess
import tempfile
import time
import uuid



def request_one(cli, prompt, model, effort, timeout):
    start = time.monotonic()
    try:
        with tempfile.TemporaryDirectory(prefix="b0_inference_") as work:
            command = [cli, "--input-format", "stream-json", "--output-format", "stream-json",
                       "--model", model, "--disable-slash-commands", "--settings", json.dumps(SETTINGS),
                       "--json-schema", json.dumps(SCHEMA), "--print-timeout", f"{timeout}s"]
            if not model.endswith("(Medium)"):
                command.extend(["--effort", effort])
            elif effort != "medium":
                raise ValueError("Medium model requires medium effort")
            completed = subprocess.run(command, input=json.dumps({"event": "user", "message": {
                "content": prompt}}) + "\n", capture_output=True, text=True, encoding="utf-8",
                errors="replace", cwd=work, timeout=timeout + 30,
                creationflags=getattr(subprocess, "CREATE_NO_WINDOW", 0))
        if completed.returncode:
            message = completed.stderr + "\n" + completed.stdout
            if classify_error(message) == "blocked":
                raise RuntimeError("Authentication or quota/credits require attention")
            raise RuntimeError(message[-4000:])
        probability, result = parse_response(completed.stdout)
        record = dict(status="success", flaky_probability=probability,
                      conversation_id=result.get("conversation_id"), usage=result.get("usage"))
    except (OSError, subprocess.SubprocessError, ValueError, RuntimeError) as exc:
        record = dict(status=classify_error(str(exc)), error=str(exc))
    record["duration_seconds"] = round(time.monotonic() - start, 3)
    return record


def run_parallel(base, selected_ids=None, n_runs=3, workers=6,
                 model="Gemini 3.8 Flash (Medium)", effort="medium",
                 experiment="b1_flash_medium_v2", timeout=300, cli=None,
                 login_before_run=False, login_handler=interactive_login, request_fn=request_one):
    base = Path(base).resolve()
    if workers < 1 or n_runs < 1:
        raise ValueError("workers and n_runs must be positive")
    if Path(experiment).name != experiment or experiment in ("", ".", ".."):
        raise ValueError("Invalid experiment folder")
    data = json.loads((base / "b1_data_v1.json").read_text(encoding="utf-8"))
    by_id = {r["id"]: r for r in data}
    ids = list(by_id) if selected_ids is None else list(dict.fromkeys(map(str, selected_ids)))
    if not ids or set(ids) - set(by_id) or len(by_id) != len(data):
        raise ValueError("Invalid/duplicate input IDs")
    prompt = (base / "prompt_b1.txt").read_text(encoding="utf-8")
    build_prompt(data, prompt, ids[0])
    cli = cli or shutil.which("agy") or str(Path.home() / "AppData/Local/agy/bin/agy.exe")
    folder = base / "b1_results" / experiment
    folder.mkdir(parents=True, exist_ok=True)
    # OS-held lock is released even after crashes. Never run two schedulers together.
    lock_file = (folder / "parallel.lock").open("a+b")
    lock_file.seek(0)
    if os.name == "nt":
        import msvcrt
        try:
            msvcrt.locking(lock_file.fileno(), msvcrt.LK_NBLCK, 1)
        except OSError:
            lock_file.close()
            raise RuntimeError("Another parallel runner already holds the experiment lock")
    try:
        config = dict(model=model, effort=effort, prompt_sha256=digest(prompt), data_sha256=digest(data),
                      settings=SETTINGS, schema=SCHEMA)
        manifest = folder / "manifest.json"
        if manifest.exists():
            old = json.loads(manifest.read_text())
            if any(old.get(k) != v for k, v in config.items()):
                raise ValueError("Prompt/model/data changed; use a new experiment")
        else:
            manifest.write_text(json.dumps(config, indent=2))
        (folder / "parallel_config.json").write_text(json.dumps(dict(workers=workers, n_runs=n_runs,
            selected_ids=ids, scheduler_sha256=digest("embedded-notebook-scheduler-v1")), indent=2))
        log = folder / "predictions.jsonl"
        previous = [json.loads(s) for s in log.read_text(encoding="utf-8").splitlines()] if log.exists() else []
        done = {(r["run"], r["id"]) for r in previous if r["status"] == "success"}
        assignments = [[(run, ident) for run in range(1, n_runs + 1) for ident in ids[w::workers]] for w in range(workers)]
        queues = [deque(job for job in jobs if job not in done) for jobs in assignments]
        (folder / "worker_assignments.json").write_text(json.dumps([
            {"worker": w + 1, "assigned": len(jobs), "pending": len(queues[w]), "jobs": jobs}
            for w, jobs in enumerate(assignments)], indent=2))
        auth_session = uuid.uuid4().hex
        paused = False
        stopped = False
        attempts = {}
        retry_at = {}
        pending = {}

        def status(state):
            payload = dict(state=state, workers=workers, active=len(pending),
                remaining=sum(map(len, queues)), completed_successes=len(done),
                updated_utc=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()))
            temp = folder / "parallel_status.tmp"
            temp.write_text(json.dumps(payload, indent=2))
            temp.replace(folder / "parallel_status.json")

        if login_before_run and not login_handler(cli, "Sign in before starting six workers."):
            status("stopped")
            return folder
        with ThreadPoolExecutor(max_workers=workers) as pool:
            while any(queues) or pending:
                if paused and not pending:
                    status("awaiting_login")
                    print("ALL WORKERS PAUSED: quota/authentication. No requests are in flight.", flush=True)
                    if not login_handler(cli, "Sign in with the desired account, then choose ready to resume all six workers."):
                        stopped = True
                        break
                    auth_session = uuid.uuid4().hex
                    paused = False
                if not paused:
                    busy = {w for w, _ in pending.values()}
                    for w, queue in enumerate(queues):
                        if queue and w not in busy and time.monotonic() >= retry_at.get(w, 0):
                            run, ident = queue[0]
                            target = build_prompt(data, prompt, ident)
                            record = dict(run=run, id=ident, worker=w + 1, model=model, effort=effort,
                                auth_session=auth_session, prompt_sha256=digest(target),
                                timestamp_utc=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()))
                            future = pool.submit(request_fn, cli, target, model, effort, timeout)
                            pending[future] = (w, record)
                status("draining_for_login" if paused else "running")
                if not pending:
                    time.sleep(0.2)
                    continue
                finished, _ = wait(pending, timeout=1, return_when=FIRST_COMPLETED)
                for future in finished:
                    w, record = pending.pop(future)
                    record.update(future.result())
                    key = (record["run"], record["id"])
                    attempts[key] = attempts.get(key, 0) + 1
                    record["attempt"] = attempts[key]
                    with log.open("a", encoding="utf-8") as f:
                        f.write(json.dumps(record, ensure_ascii=False) + "\n")
                        f.flush()
                        os.fsync(f.fileno())
                    print(f"Worker {w+1}: run {key[0]}, ID {key[1]}: {record['status']} "
                          f"{record.get('flaky_probability', '')}", flush=True)
                    if record["status"] == "blocked":
                        paused = True  # Drain existing calls; never issue new ones until re-login.
                        attempts[key] = 0
                    elif record["status"] == "transient" and attempts[key] < 3:
                        retry_at[w] = time.monotonic() + 2 ** attempts[key]
                    else:
                        queues[w].popleft()
                        if record["status"] == "success":
                            done.add(key)
            status("stopped" if stopped else "finished")
        return folder
    finally:
        lock_file.close()


# Recursive retry controller: keeps successful predictions and retries unresolved IDs.
def run_until_complete(base, ids, n_runs, workers, experiment, login_before_run):
    dataset = json.loads((base / DATA_FILE).read_text(encoding="utf-8"))
    known = {str(r["id"]) for r in dataset}
    selected = list(dict.fromkeys(map(str, ids))) if ids else [str(r["id"]) for r in dataset]
    unknown = set(selected) - known
    if unknown:
        raise ValueError(f"Unknown IDs: {sorted(unknown)}")
    expected = {(n, ident) for n in range(1, n_runs + 1) for ident in selected}
    folder = base / RESULTS_DIRNAME / experiment
    log = folder / "predictions.jsonl"
    pass_number = 0
    while True:
        records = [json.loads(line) for line in log.read_text(encoding="utf-8").splitlines() if line.strip()] if log.exists() else []
        completed = {(int(r["run"]), str(r["id"])) for r in records if r["status"] == "success"}
        remaining = expected - completed
        if not remaining:
            print(f"Complete: {len(expected)} selected predictions have valid results.")
            return folder
        pass_number += 1
        print(f"Pass {pass_number}: {len(remaining)} predictions remaining.", flush=True)
        run_parallel(base, selected_ids=selected, n_runs=n_runs, workers=workers,
                     experiment=experiment, model=MODEL, effort="medium",
                     login_before_run=login_before_run and pass_number == 1)
        state = json.loads((folder / "parallel_status.json").read_text(encoding="utf-8"))
        if state["state"] == "stopped":
            print("Stopped; rerun the cell to resume from saved results.")
            return folder
        records = [json.loads(line) for line in log.read_text(encoding="utf-8").splitlines() if line.strip()] if log.exists() else []
        completed = {(int(r["run"]), str(r["id"])) for r in records if r["status"] == "success"}
        if expected - completed:
            delay = min(60, pass_number * 10)
            print(f"Retrying remaining predictions in {delay} seconds.", flush=True)
            time.sleep(delay)

# --- B1 configuration ---
BASE = Path.cwd()
B1_IDS = []          # [] = all 300; e.g. ["198", "200"] = only these IDs
B1_N_RUNS = 3
B1_WORKERS = 8
B1_EXPERIMENT = "b1_flash_medium_v2"
MODEL = "Gemini 3.8 Flash (Medium)"
LOGIN_BEFORE_RUN = False  # True opens a login menu before the first call.
DATA_FILE = "b1_data_v1.json"
RESULTS_DIRNAME = "b1_results"
B1_RESULTS_DIR = run_until_complete(BASE, B1_IDS, B1_N_RUNS, B1_WORKERS, B1_EXPERIMENT, LOGIN_BEFORE_RUN)


In [2]:
# Compare saved predictions to actual labels. This cell makes no model calls.
from pathlib import Path
import json
import pandas as pd
from IPython.display import display
from sklearn.metrics import accuracy_score, brier_score_loss, confusion_matrix, precision_recall_fscore_support

def show_results(level, experiment, selected_ids, threshold=0.5):
    base = Path.cwd()
    folder = base / (level.lower() + "_results") / experiment
    log = folder / "predictions.jsonl"
    if not log.exists():
        raise FileNotFoundError(f"No saved predictions: {log}")
    latest = {}
    for line in log.read_text(encoding="utf-8").splitlines():
        r = json.loads(line)
        key = (int(r["run"]), str(r["id"]))
        if key not in latest or latest[key]["status"] != "success":
            latest[key] = r
    frame = pd.DataFrame(latest.values())
    frame["id"] = frame.id.astype(str)
    if selected_ids:
        frame = frame[frame.id.isin(list(map(str, selected_ids)))].copy()
    if frame.empty:
        print("No predictions found for the selected IDs.")
        return
    labels = pd.read_csv(base / "sample_300_labels.csv", dtype={"id": str})[["id", "fleaky"]].rename(columns={"fleaky": "actual"})
    frame = frame.merge(labels, on="id", how="left", validate="many_to_one")
    if frame.actual.isna().any():
        raise ValueError("A prediction has no matching label")
    if "flaky_probability" not in frame:
        frame["flaky_probability"] = float("nan")
    frame["predicted"] = frame.flaky_probability.ge(threshold).astype("Int64")
    frame.loc[frame.status != "success", "predicted"] = pd.NA
    frame["correct"] = frame.predicted.eq(frame.actual)
    rows = []
    for run, group in frame.groupby("run"):
        good = group[group.status == "success"]
        row = {"run": run, "successes": len(good), "failures": len(group)-len(good)}
        if len(good):
            y, pred = good.actual.astype(int), good.predicted.astype(int)
            precision, recall, f1, _ = precision_recall_fscore_support(y, pred, labels=[0,1], zero_division=0)
            tn, fp, fn, tp = confusion_matrix(y,pred,labels=[0,1]).ravel()
            row.update(accuracy=accuracy_score(y,pred), brier=brier_score_loss(y,good.flaky_probability),
                       flaky_precision=precision[1], flaky_recall=recall[1], flaky_f1=f1[1],
                       non_flaky_precision=precision[0], non_flaky_recall=recall[0], non_flaky_f1=f1[0],
                       tn=tn, fp=fp, fn=fn, tp=tp)
        rows.append(row)
    print(f"{level} metrics, threshold >= {threshold}; all selected tests including few-shot examples")
    display(pd.DataFrame(rows))
    print("Per-test results:")
    display(frame[["run", "id", "actual", "flaky_probability", "predicted", "correct", "status"]])
    valid = frame[frame.status == "success"]
    if len(valid):
        wide = valid.pivot(index="id", columns="run", values="flaky_probability")
        all_runs = wide.dropna()
        if len(wide.columns) >= 2 and len(all_runs):
            labels_by_run = all_runs.ge(threshold)
            agreement = labels_by_run.eq(labels_by_run.iloc[:,0],axis=0).all(axis=1)
            print(f"Same class across all available complete runs: {agreement.sum()}/{len(agreement)} ({agreement.mean():.1%})")
            print(f"Mean probability range: {(all_runs.max(axis=1)-all_runs.min(axis=1)).mean():.3f}")
        print("Probability variability by test:")
        display(valid.groupby("id").flaky_probability.agg(["count","mean","std","min","max"]))

B1_RESULTS_TO_COMPARE = "b1_flash_medium_v1"
B1_IDS = globals().get("B1_IDS", [])
B1_THRESHOLD = 0.5
show_results("B1", B1_RESULTS_TO_COMPARE, B1_IDS, B1_THRESHOLD)


B1 metrics, threshold >= 0.5; all selected tests including few-shot examples


,run,successes,failures,accuracy,brier,flaky_precision,flaky_recall,flaky_f1,non_flaky_precision,non_flaky_recall,non_flaky_f1,tn,fp,fn,tp
0,1,300,0,0.843333,0.128472,0.959677,0.739130,0.835088,0.761364,0.964029,0.850794,134,5,42,119
1,2,300,0,0.846667,0.126404,0.960000,0.745342,0.839161,0.765714,0.964029,0.853503,134,5,41,120
2,3,300,0,0.850000,0.124326,0.960317,0.751553,0.843206,0.770115,0.964029,0.856230,134,5,40,121


Per-test results:


,run,id,actual,flaky_probability,predicted,correct,status
0,1,198,0,0.95,1,False,success
1,1,200,0,0.95,1,False,success
2,1,271,0,0.90,1,False,success
3,1,272,0,0.90,1,False,success
4,1,307,0,0.03,0,True,success
...,...,...,...,...,...,...,...
895,3,410,1,0.95,1,True,success
896,3,130,1,0.85,1,True,success
897,3,207,1,0.90,1,True,success
898,3,613,1,0.15,0,False,success


Same class across all available complete runs: 297/300 (99.0%)
Mean probability range: 0.045
Probability variability by test:


,count,mean,std,min,max
id,,,,,
110,3,0.916667,0.028868,0.90,0.95
113,3,0.933333,0.028868,0.90,0.95
125,3,0.913333,0.011547,0.90,0.92
13,3,0.900000,0.000000,0.90,0.90
130,3,0.850000,0.000000,0.85,0.85
...,...,...,...,...,...
952,3,0.950000,0.000000,0.95,0.95
953,3,0.950000,0.000000,0.95,0.95
954,3,0.856667,0.040415,0.82,0.90


In [ ]:
# Run this cell only when you want to make model calls.
# Put this notebook next to the data/prompt files, and sign in to Antigravity CLI.
# IDs in code or report text remain unchanged; separate identifier fields are not sent.
"""B0 prediction and evaluation helpers. Importing this module makes no calls."""
import csv
import hashlib
import json
import math
import os
from pathlib import Path
import shutil
import subprocess
import tempfile
import time
import uuid

PRE_AUTH_FIX_SHA256 = "4ffaacc878b8617d58016be552f2874de6f7544e261cd5846fe98cb7741d06b7"


def interactive_login(cli, reason):
    """User handles OAuth in a real terminal, never in captured subprocess IO."""
    print(f"\n{reason}\nProgress is saved. No requests run while this prompt is waiting.")
    print("Choose login to open a visible Antigravity terminal. To switch accounts, "
          "enter /logout there, then sign in with the other account.")
    print("Paste any browser authorization code/link ONLY into that terminal's "
          "authorization prompt. After sign-in, exit the CLI with /exit.")
    while True:
        try:
            choice = input("Type login, ready (already signed in), or stop: ").strip().lower()
        except (EOFError, KeyboardInterrupt):
            print("Stopped. Sign in with agy in PowerShell, then rerun cell 1.")
            return False
        if choice == "stop":
            return False
        if choice == "ready":
            return True
        if choice == "login":
            if os.name == "nt":
                # Single-quoted PowerShell literal; never interpolate OAuth data.
                quoted_cli = "'" + str(cli).replace("'", "''") + "'"
                try:
                    subprocess.Popen(
                        ["powershell.exe", "-NoProfile", "-NoExit", "-Command", "& " + quoted_cli],
                        cwd=tempfile.gettempdir(), creationflags=subprocess.CREATE_NEW_CONSOLE,
                        stdin=None, stdout=None, stderr=None,
                    )
                except OSError as exc:
                    print(f"Cannot open terminal: {exc}. Open PowerShell and run agy manually.")
            else:
                print(f"Open a separate terminal and run: {cli}")
            print("Complete sign-in there, then return HERE and type ready. "
                  "If /logout closes the CLI, run agy again in that terminal.")
        else:
            print("Enter login, ready, or stop; do not paste authorization credentials here.")

SETTINGS = {
    "toolPermission": "strict", "useG1Credits": False,
    "disableSlashCommands": True, "allowNonWorkspaceAccess": False,
    "permissions": {"allow": [], "ask": [], "deny": [
        "read_url(*)", "execute_url(*)", "command(*)", "mcp(*)",
        "read_file(*)", "write_file(*)"]},
}
SCHEMA = {"type": "object", "properties": {"flaky_probability": {
    "type": "number", "minimum": 0, "maximum": 1}},
    "required": ["flaky_probability"], "additionalProperties": False}


def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False).encode()).hexdigest()


def validate_probability(value):
    if not isinstance(value, dict) or set(value) != {"flaky_probability"}:
        raise ValueError("Expected only flaky_probability")
    p = value["flaky_probability"]
    if isinstance(p, bool) or not isinstance(p, (int, float)) or not math.isfinite(p) or not 0 <= p <= 1:
        raise ValueError("Probability must be a finite number in [0,1]")
    return float(p)


def parse_response(stdout):
    events = [json.loads(line) for line in stdout.splitlines() if line.strip()]
    for event in events:
        step = event.get("step_update", {})
        tool = step.get("tool_info") or {}
        internal_finish = (step.get("step_type") == "tool" and
                           step.get("tool_name") == "finish" and
                           tool.get("name") == "finish" and
                           not step.get("subagent_info"))
        if (step.get("step_type") == "tool" or tool or step.get("subagent_info")) and not internal_finish:
            raise RuntimeError("Tool use detected; prediction rejected")
    results = [e["result"] for e in events if e.get("event") == "result"]
    if len(results) != 1:
        raise ValueError("Expected exactly one final CLI result")
    result = results[0]
    if result.get("status") != "SUCCESS":
        raise RuntimeError(str(result.get("error", result.get("status"))))
    value = result.get("structured_output")
    if value is None:
        value = json.loads(result["response"])
    return validate_probability(value), result


def classify_error(message):
    msg = message.lower()
    if any(s in msg for s in ("quota", "resource_exhausted", "429", "rate limit", "sign in", "sign-in", "logged in", "authentication", "credential", "credit", "authorization code", "unauthenticated")):
        return "blocked"
    if any(s in msg for s in ("timeout", "timed out", "503", "502", "temporarily", "connection reset")):
        return "transient"
    return "error"


def run_predictions(base, selected_ids=("198",), n_runs=1,
                    model="Gemini 3.8 Flash (Medium)", effort="medium",
                    experiment="b0_flash_medium_v1", timeout=300, cli=None,
                    login_before_run=True):
    base = Path(base).resolve()
    if not isinstance(n_runs, int) or isinstance(n_runs, bool) or n_runs < 1:
        raise ValueError("n_runs must be a positive integer")
    if Path(experiment).name != experiment or experiment in ("", ".", ".."):
        raise ValueError("experiment must be a simple folder name")
    data = json.loads((base / "b0_data_v1.json").read_text(encoding="utf-8"))
    by_id = {r["id"]: r for r in data}
    if len(by_id) != len(data):
        raise ValueError("Duplicate input IDs")
    ids = list(by_id) if selected_ids is None else list(dict.fromkeys(map(str, selected_ids)))
    if not ids or set(ids) - set(by_id):
        raise ValueError("Select existing, nonempty IDs")
    prompt = (base / "prompt_b0.txt").read_text(encoding="utf-8")
    if prompt.count("{{TEST_CODE}}") != 1:
        raise ValueError("Prompt must contain exactly one target placeholder")
    cli = cli or shutil.which("agy") or str(Path.home() / "AppData/Local/agy/bin/agy.exe")
    if not Path(cli).is_file():
        raise FileNotFoundError("Antigravity CLI not found; configure CLI_PATH")
    config = {"model": model, "effort": effort, "prompt_sha256": digest(prompt),
              "data_sha256": digest(data), "settings": SETTINGS, "schema": SCHEMA,
              "runner_sha256": hashlib.sha256(Path(__file__).read_bytes()).hexdigest()}
    folder = base / "b0_results" / experiment
    folder.mkdir(parents=True, exist_ok=True)
    manifest = folder / "manifest.json"
    if manifest.exists():
        old = json.loads(manifest.read_text())
        if old != config:
            compatible_auth_fix = (old.get("runner_sha256") == PRE_AUTH_FIX_SHA256 and
                {k: v for k, v in old.items() if k != "runner_sha256"} ==
                {k: v for k, v in config.items() if k != "runner_sha256"})
            if not compatible_auth_fix:
                raise ValueError("Experiment configuration changed: use a new EXPERIMENT name")
            (folder / "manifest_before_auth_fix.json").write_text(json.dumps(old, indent=2), encoding="utf-8")
    manifest.write_text(json.dumps(config, indent=2), encoding="utf-8")
    (folder / "prompt_snapshot.txt").write_text(prompt, encoding="utf-8")
    log = folder / "predictions.jsonl"
    previous = [json.loads(line) for line in log.read_text(encoding="utf-8").splitlines()] if log.exists() else []
    done = {(r["run"], r["id"]) for r in previous if r["status"] == "success"}
    if all((run, ident) in done for run in range(1, n_runs + 1) for ident in ids):
        return folder
    if login_before_run and not interactive_login(cli, "Confirm interactive sign-in before starting predictions."):
        return folder
    auth_session = uuid.uuid4().hex  # Local audit marker, not an account identifier.
    # One NDJSON input message per fresh process avoids Windows command-length limits.
    for run in range(1, n_runs + 1):
        for ident in ids:
            if (run, ident) in done:
                continue
            target_prompt = prompt.replace("{{TEST_CODE}}", by_id[ident]["test_code"])
            attempt = 0
            transient_failures = 0
            while True:
                attempt += 1
                record = {"run": run, "id": ident, "model": model, "effort": effort,
                          "auth_session": auth_session,
                          "prompt_sha256": digest(target_prompt), "attempt": attempt,
                          "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
                start = time.monotonic()
                try:
                    with tempfile.TemporaryDirectory(prefix="b0_inference_") as work:
                        command = [cli, "--input-format", "stream-json", "--output-format", "stream-json",
                                   "--model", model, "--disable-slash-commands",
                                   "--settings", json.dumps(SETTINGS), "--json-schema", json.dumps(SCHEMA),
                                   "--print-timeout", f"{timeout}s"]
                        if not model.endswith("(Medium)"):
                            command.extend(["--effort", effort])
                        elif effort != "medium":
                            raise ValueError("Medium model requires medium effort")
                        completed = subprocess.run(command, input=json.dumps({"event": "user", "message": {
                            "content": target_prompt}}) + "\n", capture_output=True, text=True,
                            encoding="utf-8", errors="replace", cwd=work, timeout=timeout + 30,
                            creationflags=getattr(subprocess, "CREATE_NO_WINDOW", 0))
                    if completed.returncode:
                        if classify_error(completed.stderr + "\n" + completed.stdout) == "blocked":
                            raise RuntimeError("Authentication or account quota/credits require attention")
                        raise RuntimeError((completed.stderr + "\n" + completed.stdout)[-6000:])
                    probability, result = parse_response(completed.stdout)
                    record.update(status="success", flaky_probability=probability,
                                  conversation_id=result.get("conversation_id"), usage=result.get("usage"))
                except (OSError, subprocess.SubprocessError, ValueError, RuntimeError) as exc:
                    record.update(status=classify_error(str(exc)), error=str(exc))
                record["duration_seconds"] = round(time.monotonic() - start, 3)
                with log.open("a", encoding="utf-8") as f:
                    f.write(json.dumps(record, ensure_ascii=False) + "\n")
                    f.flush()
                    os.fsync(f.fileno())
                print(f"Run {run}, ID {ident}: {record['status']} "
                      f"{record.get('flaky_probability', '')}")
                if record["status"] == "blocked":
                    if not interactive_login(cli, "Authentication failed or account quota/credits are exhausted."):
                        return folder
                    auth_session = uuid.uuid4().hex
                    transient_failures = 0
                    continue  # Retry this exact row; never discard it or restart completed work.
                if record["status"] != "transient":
                    break
                transient_failures += 1
                if transient_failures >= 3:
                    break
                time.sleep(2 ** transient_failures)
    return folder


def compare_results(base, experiment, threshold=0.5):
    import pandas as pd
    from sklearn.metrics import accuracy_score, brier_score_loss, confusion_matrix, precision_recall_fscore_support
    base = Path(base)
    folder = base / "b0_results" / experiment
    path = folder / "predictions.jsonl"
    if not path.exists():
        raise FileNotFoundError("No saved predictions. Run cell 1 when ready.")
    records = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines()]
    # Prefer a completed success; otherwise report the latest failed attempt.
    latest = {}
    for r in records:
        key = (r["run"], r["id"])
        if key not in latest or latest[key]["status"] != "success":
            latest[key] = r
    predictions = pd.DataFrame(latest.values())
    labels = pd.read_csv(base / "sample_300_labels.csv", dtype={"id": str})[["id", "fleaky"]]
    labels = labels.rename(columns={"fleaky": "actual"})
    if not labels.actual.isin([0, 1]).all():
        raise ValueError("Invalid ground-truth labels")
    table = predictions.merge(labels, on="id", how="left", validate="many_to_one")
    if table.actual.isna().any():
        raise ValueError("Prediction IDs missing from labels")
    if "flaky_probability" not in table:
        table["flaky_probability"] = float("nan")
    example_ids = {r["id"] for r in json.loads((base / "b0_few_shot.json").read_text(encoding="utf-8"))}
    table["few_shot"] = table.id.isin(example_ids)
    table["predicted"] = table.flaky_probability.ge(threshold).astype("Int64")
    table.loc[table.status != "success", "predicted"] = pd.NA
    table["correct"] = table.predicted.eq(table.actual)
    summaries = []
    for run, group in table.groupby("run"):
        for scope in ("all", "excluding_few_shot"):
            sub = group if scope == "all" else group[~group.few_shot]
            good = sub[sub.status == "success"]
            row = {"run": run, "scope": scope, "successful": len(good),
                   "failed": len(sub) - len(good), "not_attempted": len(labels) - len(group)}
            if len(good):
                y, pred = good.actual.astype(int), good.predicted.astype(int)
                precision, recall, f1, support = precision_recall_fscore_support(y, pred, labels=[0, 1], zero_division=0)
                row.update(accuracy=accuracy_score(y, pred), brier=brier_score_loss(y, good.flaky_probability))
                for k, name in enumerate(("non_flaky", "flaky")):
                    row.update({f"{name}_precision": precision[k], f"{name}_recall": recall[k],
                                f"{name}_f1": f1[k], f"{name}_support": int(support[k])})
                row.update(zip(("tn", "fp", "fn", "tp"), confusion_matrix(y, pred, labels=[0, 1]).ravel().tolist()))
            summaries.append(row)
    metrics = pd.DataFrame(summaries)
    variation = table[table.status == "success"].groupby("id").flaky_probability.agg(["count", "mean", "std", "min", "max"])
    table.to_csv(folder / "comparison.csv", index=False)
    metrics.to_csv(folder / "metrics.csv", index=False)
    variation.to_csv(folder / "probability_variability.csv")
    aggregate = metrics.groupby("scope").agg({c: ["mean", "std"] for c in metrics if c not in {"run", "scope"}})
    aggregate.to_csv(folder / "metrics_across_runs.csv")
    return table, metrics, aggregate, variation


EXAMPLES = ("516", "404")
def build_prompt(data, template, target_id):
    records = {r['id']: r for r in data}
    if len(records) != len(data):
        raise ValueError('Duplicate B2_o1 IDs')
    if template.count('{{TARGET_B2_O1_JSON}}') != 1:
        raise ValueError('Expected one target placeholder')
    if template.count('LABELED EXAMPLE ') != 2:
        raise ValueError('Expected two fixed labeled examples')
    row = records[str(target_id)]
    payload = {key: value for key, value in row.items() if key != 'id'}
    return template.replace('{{TARGET_B2_O1_JSON}}',
                            json.dumps(payload, ensure_ascii=False, indent=2), 1)

"""Six independent CLI lanes with a single checkpoint writer and sign-in gate."""
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED
from collections import deque
from pathlib import Path
import json
import os
import shutil
import subprocess
import tempfile
import time
import uuid



def request_one(cli, prompt, model, effort, timeout):
    start = time.monotonic()
    try:
        with tempfile.TemporaryDirectory(prefix="b0_inference_") as work:
            command = [cli, "--input-format", "stream-json", "--output-format", "stream-json",
                       "--model", model, "--disable-slash-commands", "--settings", json.dumps(SETTINGS),
                       "--json-schema", json.dumps(SCHEMA), "--print-timeout", f"{timeout}s"]
            if not model.endswith("(Medium)"):
                command.extend(["--effort", effort])
            elif effort != "medium":
                raise ValueError("Medium model requires medium effort")
            completed = subprocess.run(command, input=json.dumps({"event": "user", "message": {
                "content": prompt}}) + "\n", capture_output=True, text=True, encoding="utf-8",
                errors="replace", cwd=work, timeout=timeout + 30,
                creationflags=getattr(subprocess, "CREATE_NO_WINDOW", 0))
        if completed.returncode:
            message = completed.stderr + "\n" + completed.stdout
            if classify_error(message) == "blocked":
                raise RuntimeError("Authentication or quota/credits require attention")
            raise RuntimeError(message[-4000:])
        probability, result = parse_response(completed.stdout)
        record = dict(status="success", flaky_probability=probability,
                      conversation_id=result.get("conversation_id"), usage=result.get("usage"))
    except (OSError, subprocess.SubprocessError, ValueError, RuntimeError) as exc:
        record = dict(status=classify_error(str(exc)), error=str(exc))
    record["duration_seconds"] = round(time.monotonic() - start, 3)
    return record


def run_parallel(base, selected_ids=None, n_runs=3, workers=6,
                 model="Gemini 3.8 Flash (Medium)", effort="medium",
                 experiment="b2_o1_flash_medium_v1", timeout=300, cli=None,
                 login_before_run=False, login_handler=interactive_login, request_fn=request_one):
    base = Path(base).resolve()
    if workers < 1 or n_runs < 1:
        raise ValueError("workers and n_runs must be positive")
    if Path(experiment).name != experiment or experiment in ("", ".", ".."):
        raise ValueError("Invalid experiment folder")
    data = json.loads((base / "b2_o1_data_v1.json").read_text(encoding="utf-8"))
    by_id = {r["id"]: r for r in data}
    ids = list(by_id) if selected_ids is None else list(dict.fromkeys(map(str, selected_ids)))
    if not ids or set(ids) - set(by_id) or len(by_id) != len(data):
        raise ValueError("Invalid/duplicate input IDs")
    prompt = (base / "prompt_b2_o1.txt").read_text(encoding="utf-8")
    build_prompt(data, prompt, ids[0])
    cli = cli or shutil.which("agy") or str(Path.home() / "AppData/Local/agy/bin/agy.exe")
    folder = base / "b2_o1_results" / experiment
    folder.mkdir(parents=True, exist_ok=True)
    # OS-held lock is released even after crashes. Never run two schedulers together.
    lock_file = (folder / "parallel.lock").open("a+b")
    lock_file.seek(0)
    if os.name == "nt":
        import msvcrt
        try:
            msvcrt.locking(lock_file.fileno(), msvcrt.LK_NBLCK, 1)
        except OSError:
            lock_file.close()
            raise RuntimeError("Another parallel runner already holds the experiment lock")
    try:
        config = dict(model=model, effort=effort, prompt_sha256=digest(prompt), data_sha256=digest(data),
                      settings=SETTINGS, schema=SCHEMA)
        manifest = folder / "manifest.json"
        if manifest.exists():
            old = json.loads(manifest.read_text())
            if any(old.get(k) != v for k, v in config.items()):
                raise ValueError("Prompt/model/data changed; use a new experiment")
        else:
            manifest.write_text(json.dumps(config, indent=2))
        (folder / "parallel_config.json").write_text(json.dumps(dict(workers=workers, n_runs=n_runs,
            selected_ids=ids, scheduler_sha256=digest("embedded-notebook-scheduler-v1")), indent=2))
        log = folder / "predictions.jsonl"
        previous = [json.loads(s) for s in log.read_text(encoding="utf-8").splitlines()] if log.exists() else []
        done = {(r["run"], r["id"]) for r in previous if r["status"] == "success"}
        assignments = [[(run, ident) for run in range(1, n_runs + 1) for ident in ids[w::workers]] for w in range(workers)]
        queues = [deque(job for job in jobs if job not in done) for jobs in assignments]
        (folder / "worker_assignments.json").write_text(json.dumps([
            {"worker": w + 1, "assigned": len(jobs), "pending": len(queues[w]), "jobs": jobs}
            for w, jobs in enumerate(assignments)], indent=2))
        auth_session = uuid.uuid4().hex
        paused = False
        stopped = False
        attempts = {}
        retry_at = {}
        pending = {}

        def status(state):
            payload = dict(state=state, workers=workers, active=len(pending),
                remaining=sum(map(len, queues)), completed_successes=len(done),
                updated_utc=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()))
            temp = folder / "parallel_status.tmp"
            temp.write_text(json.dumps(payload, indent=2))
            temp.replace(folder / "parallel_status.json")

        if login_before_run and not login_handler(cli, "Sign in before starting six workers."):
            status("stopped")
            return folder
        with ThreadPoolExecutor(max_workers=workers) as pool:
            while any(queues) or pending:
                if paused and not pending:
                    status("awaiting_login")
                    print("ALL WORKERS PAUSED: quota/authentication. No requests are in flight.", flush=True)
                    if not login_handler(cli, "Sign in with the desired account, then choose ready to resume all six workers."):
                        stopped = True
                        break
                    auth_session = uuid.uuid4().hex
                    paused = False
                if not paused:
                    busy = {w for w, _ in pending.values()}
                    for w, queue in enumerate(queues):
                        if queue and w not in busy and time.monotonic() >= retry_at.get(w, 0):
                            run, ident = queue[0]
                            target = build_prompt(data, prompt, ident)
                            record = dict(run=run, id=ident, worker=w + 1, model=model, effort=effort,
                                auth_session=auth_session, prompt_sha256=digest(target),
                                timestamp_utc=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()))
                            future = pool.submit(request_fn, cli, target, model, effort, timeout)
                            pending[future] = (w, record)
                status("draining_for_login" if paused else "running")
                if not pending:
                    time.sleep(0.2)
                    continue
                finished, _ = wait(pending, timeout=1, return_when=FIRST_COMPLETED)
                for future in finished:
                    w, record = pending.pop(future)
                    record.update(future.result())
                    key = (record["run"], record["id"])
                    attempts[key] = attempts.get(key, 0) + 1
                    record["attempt"] = attempts[key]
                    with log.open("a", encoding="utf-8") as f:
                        f.write(json.dumps(record, ensure_ascii=False) + "\n")
                        f.flush()
                        os.fsync(f.fileno())
                    print(f"Worker {w+1}: run {key[0]}, ID {key[1]}: {record['status']} "
                          f"{record.get('flaky_probability', '')}", flush=True)
                    if record["status"] == "blocked":
                        paused = True  # Drain existing calls; never issue new ones until re-login.
                        attempts[key] = 0
                    elif record["status"] == "transient" and attempts[key] < 3:
                        retry_at[w] = time.monotonic() + 2 ** attempts[key]
                    else:
                        queues[w].popleft()
                        if record["status"] == "success":
                            done.add(key)
            status("stopped" if stopped else "finished")
        return folder
    finally:
        lock_file.close()


# Recursive retry controller: keeps successful predictions and retries unresolved IDs.
def run_until_complete(base, ids, n_runs, workers, experiment, login_before_run):
    dataset = json.loads((base / DATA_FILE).read_text(encoding="utf-8"))
    known = {str(r["id"]) for r in dataset}
    selected = list(dict.fromkeys(map(str, ids))) if ids else [str(r["id"]) for r in dataset]
    unknown = set(selected) - known
    if unknown:
        raise ValueError(f"Unknown IDs: {sorted(unknown)}")
    expected = {(n, ident) for n in range(1, n_runs + 1) for ident in selected}
    folder = base / RESULTS_DIRNAME / experiment
    log = folder / "predictions.jsonl"
    pass_number = 0
    while True:
        records = [json.loads(line) for line in log.read_text(encoding="utf-8").splitlines() if line.strip()] if log.exists() else []
        completed = {(int(r["run"]), str(r["id"])) for r in records if r["status"] == "success"}
        remaining = expected - completed
        if not remaining:
            print(f"Complete: {len(expected)} selected predictions have valid results.")
            return folder
        pass_number += 1
        print(f"Pass {pass_number}: {len(remaining)} predictions remaining.", flush=True)
        run_parallel(base, selected_ids=selected, n_runs=n_runs, workers=workers,
                     experiment=experiment, model=MODEL, effort="medium",
                     login_before_run=login_before_run and pass_number == 1)
        state = json.loads((folder / "parallel_status.json").read_text(encoding="utf-8"))
        if state["state"] == "stopped":
            print("Stopped; rerun the cell to resume from saved results.")
            return folder
        records = [json.loads(line) for line in log.read_text(encoding="utf-8").splitlines() if line.strip()] if log.exists() else []
        completed = {(int(r["run"]), str(r["id"])) for r in records if r["status"] == "success"}
        if expected - completed:
            delay = min(60, pass_number * 10)
            print(f"Retrying remaining predictions in {delay} seconds.", flush=True)
            time.sleep(delay)

# --- B2_o1 configuration ---
BASE = Path.cwd()
B2_O1_IDS = []          # [] = all 300; e.g. ["198", "200"] = only these IDs
B2_O1_N_RUNS = 3
B2_O1_WORKERS = 8
B2_O1_EXPERIMENT = "b2_o1_flash_medium_v1"
MODEL = "Gemini 3.8 Flash (Medium)"
LOGIN_BEFORE_RUN = False  # True opens a login menu before the first call.
DATA_FILE = "b2_o1_data_v1.json"
RESULTS_DIRNAME = "b2_o1_results"
B2_O1_RESULTS_DIR = run_until_complete(BASE, B2_O1_IDS, B2_O1_N_RUNS, B2_O1_WORKERS, B2_O1_EXPERIMENT, LOGIN_BEFORE_RUN)


In [3]:
# Compare saved predictions to actual labels. This cell makes no model calls.
from pathlib import Path
import json
import pandas as pd
from IPython.display import display
from sklearn.metrics import accuracy_score, brier_score_loss, confusion_matrix, precision_recall_fscore_support

def show_results(level, experiment, selected_ids, threshold=0.5):
    base = Path.cwd()
    folder = base / (level.lower() + "_results") / experiment
    log = folder / "predictions.jsonl"
    if not log.exists():
        raise FileNotFoundError(f"No saved predictions: {log}")
    latest = {}
    for line in log.read_text(encoding="utf-8").splitlines():
        r = json.loads(line)
        key = (int(r["run"]), str(r["id"]))
        if key not in latest or latest[key]["status"] != "success":
            latest[key] = r
    frame = pd.DataFrame(latest.values())
    frame["id"] = frame.id.astype(str)
    if selected_ids:
        frame = frame[frame.id.isin(list(map(str, selected_ids)))].copy()
    if frame.empty:
        print("No predictions found for the selected IDs.")
        return
    labels = pd.read_csv(base / "sample_300_labels.csv", dtype={"id": str})[["id", "fleaky"]].rename(columns={"fleaky": "actual"})
    frame = frame.merge(labels, on="id", how="left", validate="many_to_one")
    if frame.actual.isna().any():
        raise ValueError("A prediction has no matching label")
    if "flaky_probability" not in frame:
        frame["flaky_probability"] = float("nan")
    frame["predicted"] = frame.flaky_probability.ge(threshold).astype("Int64")
    frame.loc[frame.status != "success", "predicted"] = pd.NA
    frame["correct"] = frame.predicted.eq(frame.actual)
    rows = []
    for run, group in frame.groupby("run"):
        good = group[group.status == "success"]
        row = {"run": run, "successes": len(good), "failures": len(group)-len(good)}
        if len(good):
            y, pred = good.actual.astype(int), good.predicted.astype(int)
            precision, recall, f1, _ = precision_recall_fscore_support(y, pred, labels=[0,1], zero_division=0)
            tn, fp, fn, tp = confusion_matrix(y,pred,labels=[0,1]).ravel()
            row.update(accuracy=accuracy_score(y,pred), brier=brier_score_loss(y,good.flaky_probability),
                       flaky_precision=precision[1], flaky_recall=recall[1], flaky_f1=f1[1],
                       non_flaky_precision=precision[0], non_flaky_recall=recall[0], non_flaky_f1=f1[0],
                       tn=tn, fp=fp, fn=fn, tp=tp)
        rows.append(row)
    print(f"{level} metrics, threshold >= {threshold}; all selected tests including few-shot examples")
    display(pd.DataFrame(rows))
    print("Per-test results:")
    display(frame[["run", "id", "actual", "flaky_probability", "predicted", "correct", "status"]])
    valid = frame[frame.status == "success"]
    if len(valid):
        wide = valid.pivot(index="id", columns="run", values="flaky_probability")
        all_runs = wide.dropna()
        if len(wide.columns) >= 2 and len(all_runs):
            labels_by_run = all_runs.ge(threshold)
            agreement = labels_by_run.eq(labels_by_run.iloc[:,0],axis=0).all(axis=1)
            print(f"Same class across all available complete runs: {agreement.sum()}/{len(agreement)} ({agreement.mean():.1%})")
            print(f"Mean probability range: {(all_runs.max(axis=1)-all_runs.min(axis=1)).mean():.3f}")
        print("Probability variability by test:")
        display(valid.groupby("id").flaky_probability.agg(["count","mean","std","min","max"]))

B2_O1_RESULTS_TO_COMPARE = "b2_o1_flash_medium_v1"
B2_O1_IDS = globals().get("B2_O1_IDS", [])
B2_O1_THRESHOLD = 0.5
show_results("B2_o1", B2_O1_RESULTS_TO_COMPARE, B2_O1_IDS, B2_O1_THRESHOLD)

# Headline comparison: majority vote across three runs for the same 300 IDs.
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from IPython.display import display
import pandas as pd
import json

def majority_metrics(level, experiment):
    folder = Path.cwd() / (level.lower() + "_results") / experiment
    valid = {}
    for line in (folder / "predictions.jsonl").read_text(encoding="utf-8").splitlines():
        row = json.loads(line)
        if row["status"] == "success":
            valid.setdefault((int(row["run"]), str(row["id"])), row["flaky_probability"])
    labels = pd.read_csv("sample_300_labels.csv", dtype={"id": str})[["id", "fleaky"]]
    actual = labels.fleaky.to_numpy()
    ids = labels.id.tolist()
    if any((run, ident) not in valid for run in (1, 2, 3) for ident in ids):
        return {"Level": level, "Status": "Incomplete; comparison needs 900 valid predictions"}
    predicted = [int(sum(valid[(run, ident)] >= .5 for run in (1, 2, 3)) >= 2) for ident in ids]
    precision, recall, f1, _ = precision_recall_fscore_support(actual, predicted, labels=[0, 1], zero_division=0)
    tn, fp, fn, tp = confusion_matrix(actual, predicted, labels=[0, 1]).ravel()
    return {"Level": level, "Tests": len(ids), "Accuracy": accuracy_score(actual, predicted),
            "Flaky precision": precision[1], "Flaky recall": recall[1], "Flaky F1": f1[1],
            "TN": tn, "FP": fp, "FN": fn, "TP": tp}

print("B0 / B1 / B2_o1, majority vote; all 300 tests including prompt examples:")
display(pd.DataFrame([majority_metrics("B0", "b0_flash_medium_v2"),
                      majority_metrics("B1", "b1_flash_medium_v1"),
                      majority_metrics("B2_o1", "b2_o1_flash_medium_v1")]))


B2_o1 metrics, threshold >= 0.5; all selected tests including few-shot examples


,run,successes,failures,accuracy,brier,flaky_precision,flaky_recall,flaky_f1,non_flaky_precision,non_flaky_recall,non_flaky_f1,tn,fp,fn,tp
0,1,300,0,0.860000,0.113675,0.954198,0.776398,0.856164,0.786982,0.956835,0.863636,133,6,36,125
1,2,300,0,0.860000,0.116071,0.954198,0.776398,0.856164,0.786982,0.956835,0.863636,133,6,36,125
2,3,300,0,0.866667,0.105455,0.961832,0.782609,0.863014,0.792899,0.964029,0.870130,134,5,35,126


Per-test results:


,run,id,actual,flaky_probability,predicted,correct,status
0,1,198,0,0.95,1,False,success
1,2,198,0,0.95,1,False,success
2,3,198,0,0.95,1,False,success
3,1,262,0,0.00,0,True,success
4,1,200,0,0.95,1,False,success
...,...,...,...,...,...,...,...
895,3,47,1,0.95,1,True,success
896,3,130,1,0.90,1,True,success
897,3,627,1,0.20,0,False,success
898,3,485,1,0.95,1,True,success


Same class across all available complete runs: 293/300 (97.7%)
Mean probability range: 0.055
Probability variability by test:


,count,mean,std,min,max
id,,,,,
110,3,0.950000,0.000000,0.95,0.95
113,3,0.950000,0.000000,0.95,0.95
125,3,0.920000,0.000000,0.92,0.92
13,3,0.923333,0.025166,0.90,0.95
130,3,0.900000,0.000000,0.90,0.90
...,...,...,...,...,...
952,3,0.950000,0.000000,0.95,0.95
953,3,0.930000,0.017321,0.92,0.95
954,3,0.916667,0.035119,0.88,0.95


B0 / B1 / B2_o1, majority vote; all 300 tests including prompt examples:


,Level,Tests,Accuracy,Flaky precision,Flaky recall,Flaky F1,TN,FP,FN,TP
0,B0,300,0.513333,0.546584,0.546584,0.546584,66,73,73,88
1,B1,300,0.843333,0.959677,0.739130,0.835088,134,5,42,119
2,B2_o1,300,0.863333,0.961538,0.776398,0.859107,134,5,36,125
